In [1]:
import json
import warnings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore", category=DeprecationWarning)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19072\1273389939.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader = PyPDFLoader('human_predicition.pdf')
docs = loader.load()
# print(docs)

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=500
)
documents = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(documents)}")

Total chunks created: 1416


In [5]:
# This runs locally on your machine—no API keys or rate limits required!
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Create the Chroma vector store directly
db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db_hf",  # <-- Updated directory to avoid dimension conflict
)

print("Vector database created and populated successfully using local embeddings!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created and populated successfully using local embeddings!


In [10]:
# 5. Query the vector store
query = "Can you explain the step-by-step management protocol for asthma attacks?"
results = db.similarity_search_with_score(query, k=1)

# 2. Process and format results line-by-line
print("--- Formatted Step-by-Step Output ---\n")

for doc, score in results:
  content = doc.page_content

  # Split the text by newlines or markers to isolate each step
  # Cleanly strip whitespace from each line
  lines = content.split("\n")

  for line in lines:
    cleaned_line = line.strip()
    if cleaned_line:
      print(f"* {cleaned_line}")

# Optional: If you still need the raw JSON structure saved
structured_data = [
    {
        "content": doc.page_content,
        "metadata": doc.metadata,
        "score": float(score),
    }
    for doc, score in results
]
json_output = json.dumps(structured_data, indent=4)
print(json_output,'result')

--- Formatted Step-by-Step Output ---

* maintenance inhaled corticosteroids (ICS) with or without long-acting beta-agonists
* (LABA).
* Step 5: Monitoring & Action Plans: Providing a written asthma action plan and tracking
* peak flow meter logs regularly.
* 67. Non-Allergic Asthma
* Step 1: Clinical Evaluation: Spirometry confirming reversible airway obstruction with
* negative specific allergen testing.
* Step 2: Trigger Assessment: Identifying non-atopic triggers such as viral infections, cold
* air, stress, or irritants.
* Step 3: Inflammatory Profiling: Evaluating blood eosinophil counts and clinical phenotypes
* to guide targeted biologic therapy if needed.
* Step 4: Controller Medications: Initiating regular inhaled corticosteroid regimens to
* suppress chronic neutrophilic or non-atopic airway inflammation.
* Step 5: Ongoing Reassessment: Periodic pulmonary function testing and adjustment of
* controller medications.
[
    {
        "content": "maintenance inhaled corticostero

In [ ]:
# !pip install langchain-huggingface sentence-transformers